<a href="https://colab.research.google.com/github/sheryar827/fl-iot-botnet-nbaiot-cic-iot2023/blob/main/FL_IoT_Botnet_NBaIoT_MLP_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning for IoT Botnet Traffic Detection on N-BaIoT
## Comparing FedAvg, FedProx, and FedTrimmedAvg Under Varying Non-IID Conditions
### With MLP and 1D-CNN Model Architectures

**Research Question:**
> How do FedAvg, FedProx, and FedTrimmedAvg compare for IoT botnet traffic detection
> accuracy, F1-score, and convergence speed under varying non-IID conditions
> (Dirichlet α = 1.0, 0.5, 0.1) using the N-BaIoT dataset — and do the findings
> generalize across MLP and 1D-CNN architectures?

**Experimental Design:**
- 2 model architectures × 3 FL algorithms × 3 non-IID levels = 18 core experiments
- Plus 2 centralized baselines + 2 IID baselines = **22 experiments per seed**
- Multi-seed validation (3 seeds) = **66 total runs**
- Metrics: Accuracy, Macro F1, Convergence Round


## 1. Environment Setup

Install required dependencies. Designed for Google Colab.

In [ ]:
%pip install seaborn tqdm requests tabulate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Shereen\AppData\Local\Python\pythoncore-3.12-64\python.exe -m pip install --upgrade pip


In [ ]:
# ============================================================
# 1.1 — Install Dependencies
# ============================================================
#!pip install -q torch torchvision scikit-learn pandas matplotlib seaborn tqdm requests tabulate
#!apt-get update -qq && apt-get install -y -qq unar 2>/dev/null || true

import importlib
for pkg in ["torch", "sklearn", "pandas", "matplotlib", "seaborn"]:
    mod = importlib.import_module(pkg if pkg != "sklearn" else "sklearn")
    ver = getattr(mod, "__version__", "OK")
    print(f"  {pkg}: {ver}")
print("\n✅ All dependencies installed.")


  torch: 2.6.0+cu124
  sklearn: 1.8.0
  pandas: 3.0.1
  matplotlib: 3.10.8
  seaborn: 0.13.2

✅ All dependencies installed.


In [ ]:
# ============================================================
# 1.2 — Imports and Reproducibility
# ============================================================
import os, sys, json, time, copy, random, warnings, io, zipfile, glob, shutil
from pathlib import Path
from collections import OrderedDict, defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Seed:    {GLOBAL_SEED}")
print("\n✅ Imports and seeding complete.")


Device: cuda
PyTorch: 2.6.0+cu124
Seed:    42

✅ Imports and seeding complete.


## 2. Experiment Configuration

Toggle between **debug** (fast, small subset) and **full** (publication-quality) modes.

In [ ]:
# ============================================================
# 2.1 — Configuration
# ============================================================

# >>> CHANGE TO "full" FOR PUBLICATION RUNS <<<
MODE = "full"  # "debug" or "full"

if MODE == "debug":
    CFG = dict(
        max_samples_per_file=2000,
        num_rounds=10,
        local_epochs=2,
        batch_size=128,
        lr=1e-3,
        test_fraction=0.2,
        fedprox_mu=0.01,
        trimmed_fraction=0.1,
        convergence_threshold=0.95,
        dirichlet_alphas=[1.0, 0.5, 0.1],
        num_clients=9,
        seeds=[42],
        models=["MLP"],            # single model for debug
    )
    print("⚡ DEBUG mode — 1 seed, 1 model, 10 rounds. ~15 min.")
elif MODE == "full":
    CFG = dict(
        max_samples_per_file=20000,
        num_rounds=30,
        local_epochs=3,
        batch_size=512,
        lr=1e-3,
        test_fraction=0.2,
        fedprox_mu=0.01,
        trimmed_fraction=0.1,
        convergence_threshold=0.95,
        dirichlet_alphas=[1.0, 0.5, 0.1],
        num_clients=9,
        seeds=[42, 123, 7],
        models=["MLP", "CNN"],      # both architectures
    )
    n_exp = len(CFG["models"]) * 11
    n_total = n_exp * len(CFG["seeds"])
    print(f"🔬 FULL mode — {len(CFG['models'])} models × 11 experiments × "
          f"{len(CFG['seeds'])} seeds = {n_total} runs.")
    print("   ~6 hours on A100 + High-RAM.")
else:
    raise ValueError(f"Unknown MODE: {MODE}")

OUTPUT_DIR = Path("fl_nbaiot_results")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(exist_ok=True)
(OUTPUT_DIR / "csv").mkdir(exist_ok=True)
(OUTPUT_DIR / "latex").mkdir(exist_ok=True)

print(f"\nConfig: {json.dumps(CFG, indent=2)}")
print(f"Output: {OUTPUT_DIR.resolve()}")


🔬 FULL mode — 2 models × 11 experiments × 3 seeds = 66 runs.
   ~6 hours on A100 + High-RAM.

Config: {
  "max_samples_per_file": 20000,
  "num_rounds": 30,
  "local_epochs": 3,
  "batch_size": 512,
  "lr": 0.001,
  "test_fraction": 0.2,
  "fedprox_mu": 0.01,
  "trimmed_fraction": 0.1,
  "convergence_threshold": 0.95,
  "dirichlet_alphas": [
    1.0,
    0.5,
    0.1
  ],
  "num_clients": 9,
  "seeds": [
    42,
    123,
    7
  ],
  "models": [
    "MLP",
    "CNN"
  ]
}
Output: C:\Windows\System32\fl_nbaiot_results


## 3. N-BaIoT Dataset Download and Preparation

The N-BaIoT dataset contains network traffic features from 9 IoT devices, capturing benign and botnet attack traffic (Mirai and Gafgyt variants).

**Reference:** Meidan et al., "N-BaIoT: Network-Based Detection of IoT Botnet Attacks Using Deep Autoencoders," *IEEE Pervasive Computing*, 2018.

In [ ]:
# ============================================================
# 3.1 — Download and Extract N-BaIoT (WINDOWS-FRIENDLY)
# ============================================================
import requests, subprocess, zipfile, glob, os, sys
from pathlib import Path

# Automatically install patool for Windows if it is missing
try:
    import patoolib
except ImportError:
    print("Installing 'patool' for Windows .rar extraction...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "patool"])
    import patoolib

DATA_DIR = Path("nbaiot_data")
DATA_DIR.mkdir(exist_ok=True)

NBAIOT_URL = "https://archive.ics.uci.edu/static/public/442/detection+of+iot+botnet+attacks+n+baiot.zip"

zip_path = DATA_DIR / "nbaiot.zip"
if not zip_path.exists():
    print("Downloading N-BaIoT dataset (~90 MB)...")
    resp = requests.get(NBAIOT_URL, stream=True, timeout=300)
    resp.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Already exists: {zip_path}")

# --- Extract ---
extract_dir = DATA_DIR / "extracted"
if not extract_dir.exists():
    extract_dir.mkdir(parents=True)
    print("Extracting main archive...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    # Handle nested zip files
    for nested in glob.glob(str(extract_dir / "**/*.zip"), recursive=True):
        nested_dest = Path(nested).parent / Path(nested).stem
        nested_dest.mkdir(exist_ok=True)
        try:
            with zipfile.ZipFile(nested, "r") as zf2:
                zf2.extractall(nested_dest)
        except zipfile.BadZipFile:
            pass

    # Handle nested rar files natively on Windows
    rar_files = glob.glob(str(extract_dir / "**/*.rar"), recursive=True)
    if rar_files:
        print(f"Found {len(rar_files)} .rar files. Extracting attack data...")
        for rar_path in rar_files:
            rar_dest = Path(rar_path).parent / Path(rar_path).stem
            rar_dest.mkdir(exist_ok=True)
            try:
                patoolib.extract_archive(rar_path, outdir=str(rar_dest), interactive=False)
            except Exception as e:
                print(f"  ⚠️ Windows .rar extraction issue: {e}")
                print("  -> IMPORTANT: Make sure you have 7-Zip or WinRAR installed on your PC!")

    print("Extraction complete.")
else:
    print(f"Already extracted: {extract_dir}")

# --- Always define all_csvs ---
all_csvs = sorted(glob.glob(str(extract_dir / "**/*.csv"), recursive=True))

# Diagnostic summary
print(f"\nTotal CSV files: {len(all_csvs)}")
benign_count = sum(1 for c in all_csvs if "benign" in c.lower())
attack_count = sum(1 for c in all_csvs if "mirai" in c.lower() or "gafgyt" in c.lower())
print(f"  Benign CSVs: {benign_count}")
print(f"  Attack CSVs: {attack_count}")

if len(all_csvs) == 0:
    raise RuntimeError("No CSV files found after extraction! Check download.")

# Show sample paths
print("\nSample paths:")
for c in all_csvs[:10]:
    rel = os.path.relpath(c, extract_dir)
    print(f"  {rel}")
if len(all_csvs) > 10:
    print(f"  ... and {len(all_csvs)-10} more.")

Already exists: nbaiot_data\nbaiot.zip
Already extracted: nbaiot_data\extracted

Total CSV files: 90
  Benign CSVs: 9
  Attack CSVs: 80

Sample paths:
  Danmini_Doorbell\benign_traffic.csv
  Danmini_Doorbell\gafgyt_attacks\combo.csv
  Danmini_Doorbell\gafgyt_attacks\junk.csv
  Danmini_Doorbell\gafgyt_attacks\scan.csv
  Danmini_Doorbell\gafgyt_attacks\tcp.csv
  Danmini_Doorbell\gafgyt_attacks\udp.csv
  Danmini_Doorbell\mirai_attacks\ack.csv
  Danmini_Doorbell\mirai_attacks\scan.csv
  Danmini_Doorbell\mirai_attacks\syn.csv
  Danmini_Doorbell\mirai_attacks\udp.csv
  ... and 80 more.


In [ ]:
# ============================================================
# 3.2 — Parse CSV Files into Unified Dataset
# ============================================================

# Ensure all_csvs is defined (in case cell 3.1 was cached)
DATA_DIR = Path("nbaiot_data")
extract_dir = DATA_DIR / "extracted"
all_csvs = sorted(glob.glob(str(extract_dir / "**/*.csv"), recursive=True))
print(f"CSV files available for parsing: {len(all_csvs)}")
if len(all_csvs) == 0:
    raise RuntimeError("No CSVs found. Run cell 3.1 (download/extract) first.")


def parse_nbaiot_csvs(csv_paths, max_samples=None):
    """Parse N-BaIoT CSVs. Infer device and label from the full path."""
    frames = []
    label_counts = defaultdict(int)
    device_counts = defaultdict(int)

    known_devices = [
        "danmini_doorbell", "ecobee_thermostat", "ennio_doorbell",
        "philips_b120n10_baby_monitor", "provision_pt_737e_security_camera",
        "provision_pt_838_security_camera", "samsung_snh_1011_n_webcam",
        "simplehome_xcs7_1002_wht_security_camera",
        "simplehome_xcs7_1003_wht_security_camera",
    ]

    mirai_types = ["ack", "scan", "syn", "udpplain", "udp"]
    gafgyt_types = ["combo", "junk", "scan", "tcp", "udp"]

    for path in csv_paths:
        norm = str(path).lower().replace(" ", "_").replace("-", "_")
        parts = [p.lower().replace(" ", "_").replace("-", "_") for p in Path(path).parts]
        fname = Path(path).stem.lower().replace(" ", "_").replace("-", "_")

        # --- Device ---
        device = "unknown"
        for d in known_devices:
            if d in norm:
                device = d
                break

        # --- Label ---
        label = "unknown"
        if "benign" in norm:
            label = "benign"
        else:
            is_mirai = "mirai" in norm
            is_gafgyt = "gafgyt" in norm or "bashlite" in norm

            if is_mirai:
                for atk in mirai_types:
                    if atk in fname or any(atk == p for p in parts):
                        label = f"mirai_{atk}"
                        break
                if label == "unknown":
                    label = "mirai_other"
            elif is_gafgyt:
                for atk in gafgyt_types:
                    if atk in fname or any(atk == p for p in parts):
                        label = f"gafgyt_{atk}"
                        break
                if label == "unknown":
                    label = "gafgyt_other"

        if label == "unknown" or device == "unknown":
            continue

        try:
            df = pd.read_csv(path, low_memory=False)
            if df.shape[1] < 10:
                continue
            if max_samples and len(df) > max_samples:
                df = df.sample(n=max_samples, random_state=GLOBAL_SEED)
            df["device"] = device
            df["label"] = label
            frames.append(df)
            label_counts[label] += len(df)
            device_counts[device] += len(df)
        except Exception as e:
            print(f"  ⚠ Skip {path}: {e}")

    if not frames:
        raise RuntimeError("No data loaded!")
    combined = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(combined)} samples from {len(frames)} files.")
    print(f"Labels: {dict(label_counts)}")
    print(f"Devices: {dict(device_counts)}")
    return combined


raw_df = parse_nbaiot_csvs(all_csvs, max_samples=CFG["max_samples_per_file"])
print(f"\nShape: {raw_df.shape}")
print(f"Unique labels: {raw_df['label'].nunique()}")
print(raw_df["label"].value_counts())

# ============================================================
# FALLBACK: If only benign found, use DEVICE as the classification target
# ============================================================
# The UCI N-BaIoT archive sometimes ships only benign traffic CSVs.
# Attack CSVs may be in .rar archives that require manual download.
# In that case, we reframe the task as multi-class DEVICE IDENTIFICATION
# (still a valid and published FL-IoT research task).

USE_DEVICE_AS_CLASS = False

if raw_df["label"].nunique() < 2:
    print("\n" + "=" * 60)
    print("  ⚠️  ONLY BENIGN TRAFFIC FOUND IN DATASET")
    print("=" * 60)
    print("  The UCI N-BaIoT archive may not include attack CSVs directly.")
    print("  Attack data is sometimes in .rar archives needing manual extraction.")
    print()
    print("  APPLYING FALLBACK: Using DEVICE TYPE as classification target.")
    print("  This reframes the task as IoT device identification — a valid")
    print("  federated learning research problem with the same N-BaIoT features.")
    print("  The 9 devices become 9 classes; Dirichlet non-IID still applies.")
    print("=" * 60)

    USE_DEVICE_AS_CLASS = True
    raw_df["label"] = raw_df["device"]
    print(f"\nNew label distribution ({raw_df['label'].nunique()} classes):")
    print(raw_df["label"].value_counts())


CSV files available for parsing: 90
Loaded 1772641 samples from 89 files.
Labels: {'benign': 172641, 'gafgyt_combo': 180000, 'gafgyt_junk': 180000, 'gafgyt_scan': 180000, 'gafgyt_tcp': 180000, 'gafgyt_udp': 180000, 'mirai_ack': 140000, 'mirai_scan': 140000, 'mirai_syn': 140000, 'mirai_udp': 140000, 'mirai_udpplain': 140000}
Devices: {'danmini_doorbell': 220000, 'ecobee_thermostat': 213113, 'ennio_doorbell': 120000, 'philips_b120n10_baby_monitor': 220000, 'provision_pt_737e_security_camera': 220000, 'provision_pt_838_security_camera': 220000, 'samsung_snh_1011_n_webcam': 120000, 'simplehome_xcs7_1002_wht_security_camera': 220000, 'simplehome_xcs7_1003_wht_security_camera': 219528}

Shape: (1772641, 117)
Unique labels: 11
label
gafgyt_combo      180000
gafgyt_junk       180000
gafgyt_scan       180000
gafgyt_tcp        180000
gafgyt_udp        180000
benign            172641
mirai_ack         140000
mirai_scan        140000
mirai_syn         140000
mirai_udp         140000
mirai_udpplain

## 4. Preprocessing

Clean features, encode labels, normalize, and create train/test splits.

In [ ]:
# ============================================================
# 4.1 — Clean, Encode, Normalize, Split
# ============================================================

device_col = raw_df["device"].values
label_col = raw_df["label"].values

feature_df = raw_df.drop(columns=["device", "label"], errors="ignore")
feature_df = feature_df.apply(pd.to_numeric, errors="coerce")
feature_df.replace([np.inf, -np.inf], np.nan, inplace=True)
feature_df.fillna(0, inplace=True)

X = feature_df.values.astype(np.float32)
NUM_FEATURES = X.shape[1]
print(f"Features: {NUM_FEATURES}")

le = LabelEncoder()
y = le.fit_transform(label_col)
NUM_CLASSES = len(le.classes_)
print(f"Classes ({NUM_CLASSES}): {list(le.classes_)}")

device_le = LabelEncoder()
client_ids = device_le.fit_transform(device_col)
print(f"Devices ({len(device_le.classes_)}): {list(device_le.classes_)}")

scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

X_train, X_test, y_train, y_test, cid_train, cid_test = train_test_split(
    X, y, client_ids,
    test_size=CFG["test_fraction"],
    random_state=GLOBAL_SEED,
    stratify=y
)

print(f"\nTrain: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

test_dataset = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=CFG["batch_size"], shuffle=False)

print("\n✅ Preprocessing complete.")

Features: 115
Classes (11): ['benign', 'gafgyt_combo', 'gafgyt_junk', 'gafgyt_scan', 'gafgyt_tcp', 'gafgyt_udp', 'mirai_ack', 'mirai_scan', 'mirai_syn', 'mirai_udp', 'mirai_udpplain']
Devices (9): ['danmini_doorbell', 'ecobee_thermostat', 'ennio_doorbell', 'philips_b120n10_baby_monitor', 'provision_pt_737e_security_camera', 'provision_pt_838_security_camera', 'samsung_snh_1011_n_webcam', 'simplehome_xcs7_1002_wht_security_camera', 'simplehome_xcs7_1003_wht_security_camera']

Train: 1418112  |  Test: 354529

✅ Preprocessing complete.


## 5. Federated Data Partitioning

Three schemes: IID (uniform random), and Dirichlet non-IID at α ∈ {1.0, 0.5, 0.1}.

In [ ]:
# ============================================================
# 5.1 — Partitioning Functions
# ============================================================

def partition_iid(X, y, num_clients, seed=42):
    rng = np.random.RandomState(seed)
    indices = rng.permutation(len(X))
    splits = np.array_split(indices, num_clients)
    return {i: s.astype(np.int64) for i, s in enumerate(splits)}


def partition_dirichlet(y, num_clients, alpha, seed=42):
    """Dirichlet non-IID partition. Ensures all clients get at least 1 sample."""
    rng = np.random.RandomState(seed)
    num_classes = len(np.unique(y))
    client_indices = {cid: [] for cid in range(num_clients)}  # explicit init

    for c in range(num_classes):
        class_idx = np.where(y == c)[0]
        rng.shuffle(class_idx)
        proportions = rng.dirichlet(np.repeat(alpha, num_clients))
        proportions = np.maximum(proportions, 1e-6)
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * len(class_idx)).astype(int)[:-1]
        for cid, chunk in enumerate(np.split(class_idx, splits)):
            client_indices[cid].extend(chunk.tolist())

    # Ensure every client has at least 1 sample (steal from largest if needed)
    for cid in range(num_clients):
        if len(client_indices[cid]) == 0:
            # Find client with the most samples and donate one
            donor = max(range(num_clients), key=lambda c: len(client_indices[c]))
            if client_indices[donor]:
                client_indices[cid].append(client_indices[donor].pop())

    # Shuffle within each client and convert to int64 arrays
    for cid in range(num_clients):
        rng.shuffle(client_indices[cid])
        client_indices[cid] = np.array(client_indices[cid], dtype=np.int64)

    return client_indices


def summarize_partition(partition, y, label_names, title=""):
    rows = []
    for cid in sorted(partition.keys()):
        idx = partition[cid]
        if len(idx) == 0:
            row = {"client": cid, "total": 0}
            for ln in label_names:
                row[ln] = 0
        else:
            labels = y[idx]
            counts = dict(zip(*np.unique(labels, return_counts=True)))
            row = {"client": cid, "total": len(idx)}
            for li, ln in enumerate(label_names):
                row[ln] = counts.get(li, 0)
        rows.append(row)
    df = pd.DataFrame(rows)
    if title:
        print(f"\n--- {title} ---")
    print(df.to_string(index=False))
    return df

print("✅ Partitioning functions defined.")


✅ Partitioning functions defined.


In [ ]:
# ============================================================
# 5.2 — Create and Visualize Partitions
# ============================================================

label_names = list(le.classes_)
partitions = {}
partitions["IID"] = partition_iid(X_train, y_train, CFG["num_clients"], seed=GLOBAL_SEED)
for alpha in CFG["dirichlet_alphas"]:
    partitions[f"Dir_a{alpha}"] = partition_dirichlet(
        y_train, CFG["num_clients"], alpha, seed=GLOBAL_SEED
    )

for name, part in partitions.items():
    df_p = summarize_partition(part, y_train, label_names, title=name)
    df_p.to_csv(OUTPUT_DIR / "csv" / f"partition_{name}.csv", index=False)

# Heatmap visualization
fig, axes = plt.subplots(1, len(partitions), figsize=(5 * len(partitions), 5))
if len(partitions) == 1:
    axes = [axes]
for ax, (name, part) in zip(axes, partitions.items()):
    dist = np.zeros((CFG["num_clients"], NUM_CLASSES))
    for cid in sorted(part.keys()):
        labels = y_train[part[cid]]
        for li in range(NUM_CLASSES):
            dist[cid, li] = np.sum(labels == li)
    row_sums = dist.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    dist_norm = dist / row_sums
    sns.heatmap(dist_norm, ax=ax, cmap="YlOrRd", vmin=0, vmax=1,
                xticklabels=[ln[:10] for ln in label_names],
                yticklabels=[f"C{i}" for i in range(CFG["num_clients"])])
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Class")
    ax.set_ylabel("Client")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "partition_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Partition heatmaps saved.")


--- IID ---
 client  total  benign  gafgyt_combo  gafgyt_junk  gafgyt_scan  gafgyt_tcp  gafgyt_udp  mirai_ack  mirai_scan  mirai_syn  mirai_udp  mirai_udpplain
      0 157568   15492         15990        16033        16004       16068       15979      12416       12475      12374      12362           12375
      1 157568   15362         15950        15797        15839       16206       16110      12506       12407      12374      12568           12449
      2 157568   15198         15831        16030        16080       16080       15943      12491       12399      12554      12589           12373
      3 157568   15407         16073        16025        15810       15812       15941      12396       12559      12535      12402           12608
      4 157568   15123         15964        15893        16230       15990       15935      12749       12430      12445      12326           12483
      5 157568   15470         16088        15947        15898       16012       15996      12335  

## 6. Model Architectures

Two architectures are evaluated to test whether findings generalize:

**MLP (Multi-Layer Perceptron):** 4 hidden layers (128→64→64→32), ReLU, LayerNorm, Dropout. ~20K params. The standard baseline used by most FL-IoT papers.

**1D-CNN:** Two Conv1D layers (1→32 k=5, 32→64 k=3) followed by AdaptiveAvgPool1d and a small classifier head. ~12K params. Captures local correlations in N-BaIoT's 115 statistical features.

Both use **LayerNorm** instead of BatchNorm because BatchNorm's running statistics (mean/var) accumulate locally on each client and do NOT aggregate meaningfully under FedAvg. This causes the global model to produce near-random predictions when evaluated server-side. LayerNorm normalizes per-sample and has no running statistics, making it fully compatible with federated aggregation.


In [ ]:
# ============================================================
# 6.1 — Model Definitions (MLP + 1D-CNN, FL-Compatible)
# ============================================================

class IoTBotnetMLP(nn.Module):
    """
    4-hidden-layer MLP for N-BaIoT traffic classification.
    LayerNorm instead of BatchNorm for FL compatibility.
    """
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.LayerNorm(64),
            nn.Dropout(0.2),

            nn.Linear(64, 64),
            nn.ReLU(),
            nn.LayerNorm(64),
            nn.Dropout(0.1),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.LayerNorm(32),

            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class IoTBotnetCNN(nn.Module):
    """
    Lightweight 1D-CNN for N-BaIoT traffic classification.

    Input (115) → reshape (1,115) → Conv1D(1→32,k=5) → ReLU →
    Conv1D(32→64,k=3) → ReLU → AdaptiveAvgPool1d(1) → Flatten →
    Dense(64→32) → ReLU → LayerNorm → Dense(32→C)

    Kernel sizes 5 and 3 capture local correlations in the 115
    statistical features (stream aggregation windows in N-BaIoT).
    LayerNorm (not BatchNorm) for FL compatibility.
    ~12K parameters — lightweight enough for IoT client simulation.
    """
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.reshape = nn.Unflatten(1, (1, input_dim))
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.LayerNorm(32),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        x = self.reshape(x)
        x = self.features(x)
        return self.classifier(x)


# ---- Model selector ----
CURRENT_MODEL = "MLP"

def get_model():
    if CURRENT_MODEL == "MLP":
        return IoTBotnetMLP(NUM_FEATURES, NUM_CLASSES).to(DEVICE)
    elif CURRENT_MODEL == "CNN":
        return IoTBotnetCNN(NUM_FEATURES, NUM_CLASSES).to(DEVICE)
    else:
        raise ValueError(f"Unknown model: {CURRENT_MODEL}")


def get_parameters(model):
    return [val.cpu().numpy() for val in model.state_dict().values()]


def set_parameters(model, parameters):
    state_dict = OrderedDict(
        {k: torch.tensor(v, dtype=torch.float32) for k, v in
         zip(model.state_dict().keys(), parameters)}
    )
    model.load_state_dict(state_dict, strict=True)


# Verify both models
for name, cls in [("MLP", IoTBotnetMLP), ("CNN", IoTBotnetCNN)]:
    _m = cls(NUM_FEATURES, NUM_CLASSES).to(DEVICE)
    _n = sum(p.numel() for p in _m.parameters())
    _o = _m(torch.randn(2, NUM_FEATURES).to(DEVICE))
    assert _o.shape == (2, NUM_CLASSES)
    print(f"  {name}: {_n:,} params | output {_o.shape} ✓")
    del _m

print("\n✅ Both models defined and verified (LayerNorm, FL-compatible).")


  MLP: 30,283 params | output torch.Size([2, 11]) ✓
  CNN: 8,907 params | output torch.Size([2, 11]) ✓

✅ Both models defined and verified (LayerNorm, FL-compatible).


In [ ]:
# ============================================================
# 6.2 — Training and Evaluation Utilities
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, device,
                    proximal_mu=0.0, global_params=None):
    model.train()
    total_loss, total_n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        if proximal_mu > 0.0 and global_params is not None:
            prox = sum(((lp - gp) ** 2).sum()
                       for lp, gp in zip(model.parameters(), global_params))
            loss = loss + (proximal_mu / 2.0) * prox
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
        total_n += len(yb)
    return total_loss / max(total_n, 1)


def evaluate_model(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    preds_all, labels_all = [], []
    total_loss, total_n = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total_loss += criterion(out, yb).item() * len(yb)
            total_n += len(yb)
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(yb.cpu().numpy())
    acc = accuracy_score(labels_all, preds_all)
    f1 = f1_score(labels_all, preds_all, average="macro", zero_division=0)
    return total_loss / max(total_n, 1), acc, f1


def convergence_round(f1_list, threshold=None):
    if threshold is None:
        threshold = CFG["convergence_threshold"]
    best = max(f1_list) if f1_list else 0
    target = threshold * best
    for i, v in enumerate(f1_list):
        if v >= target:
            return i + 1
    return len(f1_list)

print("✅ Training utilities defined.")

✅ Training utilities defined.


## 7. Centralized Baseline

Train the MLP on the full training set (no federation) to establish an upper-bound benchmark.

In [ ]:
# ============================================================
# 7.1 — Centralized Baseline (runs inside experiment loop)
# ============================================================
# The centralized baseline is now executed per-seed inside cell 9.2.
# This ensures all seeds are covered with mean ± std reporting.
print("ℹ️  Centralized baseline runs per-seed in the experiment loop (cell 9.2).")
print("   No standalone execution needed here.")


ℹ️  Centralized baseline runs per-seed in the experiment loop (cell 9.2).
   No standalone execution needed here.


## 8. Federated Learning Implementation

We implement a **manual FL simulation loop** for full transparency and reproducibility. This gives complete control over client training, parameter aggregation, and server-side evaluation — every step is visible and debuggable.

Three aggregation strategies are implemented from scratch:
1. **FedAvg** (McMahan et al., 2017) — weighted average of client model parameters, proportional to local dataset size.
2. **FedProx** (Li et al., 2020) — FedAvg aggregation + proximal regularization during local training. The term μ/2 · ‖w - w_global‖² penalizes client drift.
3. **FedTrimmedAvg** (Yin et al., 2018) — coordinate-wise trimmed mean. For each parameter element, sorts values across clients, discards the top and bottom fraction, then averages the rest. Robust to outlier clients.

Each round: broadcast global parameters → clients train locally → aggregate → evaluate on global test set → record metrics.

In [ ]:
# ============================================================
# 8.1 — Aggregation Functions
# ============================================================

def aggregate_fedavg(client_params_list, client_sizes):
    """
    FedAvg: weighted average of client parameters.
    client_params_list: list of [list of np.ndarray] per client
    client_sizes: list of int (number of samples per client)
    """
    total = sum(client_sizes)
    weights = [n / total for n in client_sizes]
    num_layers = len(client_params_list[0])
    aggregated = []
    for layer_i in range(num_layers):
        weighted_sum = np.zeros_like(client_params_list[0][layer_i])
        for c, w in enumerate(weights):
            weighted_sum += w * client_params_list[c][layer_i]
        aggregated.append(weighted_sum)
    return aggregated


def aggregate_trimmed_avg(client_params_list, trim_fraction=0.1):
    """
    FedTrimmedAvg: coordinate-wise trimmed mean.
    Sorts each parameter element across clients, trims extremes, averages.
    """
    n_clients = len(client_params_list)
    trim_n = max(1, int(trim_fraction * n_clients))
    num_layers = len(client_params_list[0])
    aggregated = []
    for layer_i in range(num_layers):
        stacked = np.stack([c[layer_i] for c in client_params_list], axis=0)
        sorted_v = np.sort(stacked, axis=0)
        if n_clients > 2 * trim_n:
            trimmed = sorted_v[trim_n:-trim_n]
        else:
            trimmed = sorted_v
        aggregated.append(np.mean(trimmed, axis=0))
    return aggregated


print("✅ Aggregation functions: FedAvg, FedTrimmedAvg")


✅ Aggregation functions: FedAvg, FedTrimmedAvg


### 8.2 Local Training and FL Simulation Loop

In [ ]:
# ============================================================
# 8.2 — Local Training Function
# ============================================================

def train_client(global_params, X_local, y_local, local_epochs,
                 lr, batch_size, device, proximal_mu=0.0):
    """
    Simulate one FL client's local training.
    Returns updated parameters and number of training samples.
    """
    model = get_model()
    set_parameters(model, global_params)

    # For FedProx: freeze a copy of global parameters
    if proximal_mu > 0:
        global_param_tensors = [p.clone().detach() for p in model.parameters()]
    else:
        global_param_tensors = None

    dataset = TensorDataset(
        torch.tensor(X_local, dtype=torch.float32),
        torch.tensor(y_local, dtype=torch.long)
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for _ in range(local_epochs):
        train_one_epoch(model, loader, optimizer, criterion, device,
                        proximal_mu=proximal_mu, global_params=global_param_tensors)

    return get_parameters(model), len(dataset)


print("✅ Local training function defined.")


✅ Local training function defined.


## 9. Experiment Execution
Run all 22 experiments per seed: 2 models × (1 centralized + 1 IID + 9 non-IID).
With 3 seeds: 22 × 3 = 66 total runs.


In [ ]:
# ============================================================
# 9.1 — Federated Experiment Runner (explicit loop, seeded)
# ============================================================

def set_global_seed(seed):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_fl_experiment(run_name, algorithm, partition, alpha_label,
                      proximal_mu=0.0, trim_fraction=0.1, seed=42):
    """
    Run a single FL experiment with explicit training loop.
    Accepts seed for reproducible multi-run experiments.
    """
    set_global_seed(seed)

    print(f"\n{'='*60}")
    print(f"  {run_name}  |  {algorithm}  |  α={alpha_label}  |  seed={seed}")
    print(f"{'='*60}")

    num_clients = CFG["num_clients"]
    num_rounds = CFG["num_rounds"]
    local_epochs = CFG["local_epochs"]

    # Initialize global model
    global_model = get_model()
    global_params = get_parameters(global_model)

    # Evaluate initial model (round 0)
    loss0, acc0, f10 = evaluate_model(global_model, test_loader, DEVICE)
    print(f"  Round  0 (init) |  Acc: {acc0:.4f}  F1: {f10:.4f}  Loss: {loss0:.4f}")

    round_history = []

    for rnd in range(1, num_rounds + 1):
        # ---- Client local training ----
        client_results = []
        for cid in range(num_clients):
            idx = partition[cid]
            if len(idx) == 0:
                continue
            mu = proximal_mu if algorithm == "FedProx" else 0.0
            updated_params, n_samples = train_client(
                global_params,
                X_train[idx],
                y_train[idx],
                local_epochs=local_epochs,
                lr=CFG["lr"],
                batch_size=CFG["batch_size"],
                device=DEVICE,
                proximal_mu=mu,
            )
            client_results.append((updated_params, n_samples))

        if not client_results:
            print(f"  Round {rnd:2d}  |  ⚠ No client results!")
            continue

        # ---- Aggregation ----
        client_params_list = [params for params, _ in client_results]
        client_sizes = [n_samples for _, n_samples in client_results]

        if algorithm in ("FedAvg", "FedProx"):
            global_params = aggregate_fedavg(client_params_list, client_sizes)
        elif algorithm == "FedTrimmedAvg":
            global_params = aggregate_trimmed_avg(client_params_list, trim_fraction)
        else:
            raise ValueError(f"Unknown algorithm: {algorithm}")

        # ---- Update global model and evaluate ----
        set_parameters(global_model, global_params)
        loss, acc, f1 = evaluate_model(global_model, test_loader, DEVICE)
        round_history.append(dict(round=rnd, accuracy=acc, f1=f1, loss=loss))

        if rnd % max(1, num_rounds // 6) == 0 or rnd == 1 or rnd == num_rounds:
            print(f"  Round {rnd:2d}/{num_rounds}  |  "
                  f"Acc: {acc:.4f}  F1: {f1:.4f}  Loss: {loss:.4f}")

    # ---- Collect results ----
    if not round_history:
        accs, f1s = [0.0], [0.0]
    else:
        accs = [m["accuracy"] for m in round_history]
        f1s = [m["f1"] for m in round_history]

    result = dict(
        run_name=run_name, algorithm=algorithm, alpha=alpha_label, seed=seed,
        final_acc=accs[-1], best_acc=max(accs),
        final_f1=f1s[-1], best_f1=max(f1s),
        convergence_round=convergence_round(f1s),
        history=dict(round=list(range(1, len(accs) + 1)), accuracy=accs, f1=f1s),
    )

    print(f"  ► Final Acc: {result['final_acc']:.4f}  F1: {result['final_f1']:.4f}")
    print(f"  ► Best  Acc: {result['best_acc']:.4f}  F1: {result['best_f1']:.4f}")
    print(f"  ► Convergence round: {result['convergence_round']}")

    # Save per-run history
    safe = f"{run_name}_s{seed}".replace(" ", "_").replace("=", "").replace("α", "a")
    pd.DataFrame(result["history"]).to_csv(
        OUTPUT_DIR / "csv" / f"history_{safe}.csv", index=False)

    return result


def run_centralized(seed=42):
    """Run centralized baseline with a given seed."""
    set_global_seed(seed)
    print(f"\n{'='*50}")
    print(f"  CENTRALIZED BASELINE  |  seed={seed}")
    print(f"{'='*50}")
    model = get_model()
    optimizer = optim.Adam(model.parameters(), lr=CFG["lr"])
    criterion = nn.CrossEntropyLoss()
    train_ds = TensorDataset(X_train_t, y_train_t)
    train_ld = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True)
    num_epochs = CFG["num_rounds"] * CFG["local_epochs"]
    hist = {"epoch": [], "train_loss": [], "test_loss": [], "test_acc": [], "test_f1": []}
    for ep in range(1, num_epochs + 1):
        tl = train_one_epoch(model, train_ld, optimizer, criterion, DEVICE)
        vl, va, vf = evaluate_model(model, test_loader, DEVICE)
        hist["epoch"].append(ep)
        hist["train_loss"].append(tl)
        hist["test_loss"].append(vl)
        hist["test_acc"].append(va)
        hist["test_f1"].append(vf)
        if ep % max(1, num_epochs // 5) == 0 or ep == 1:
            print(f"  Epoch {ep:3d}/{num_epochs}  Loss:{vl:.4f}  Acc:{va:.4f}  F1:{vf:.4f}")
    pd.DataFrame(hist).to_csv(OUTPUT_DIR / "csv" / f"centralized_history_s{seed}.csv", index=False)
    return dict(
        run_name="Centralized", algorithm="Centralized", alpha="-", seed=seed,
        final_acc=hist["test_acc"][-1], best_acc=max(hist["test_acc"]),
        final_f1=hist["test_f1"][-1], best_f1=max(hist["test_f1"]),
        convergence_round=convergence_round(hist["test_f1"]),
        history=dict(round=hist["epoch"], accuracy=hist["test_acc"], f1=hist["test_f1"]),
    )


print("✅ Experiment runner defined (multi-seed support).")


✅ Experiment runner defined (multi-seed support).


In [ ]:
# ============================================================
# 9.2 — Execute All Experiments × All Models × All Seeds
# ============================================================

all_results = []  # list of dicts, one per (model, experiment, seed)

seeds = CFG["seeds"]
models = CFG["models"]
n_exp_per_seed = len(models) * 11
n_total = n_exp_per_seed * len(seeds)
print(f"Running {len(models)} model(s) × {len(seeds)} seed(s) × 11 experiments = {n_total} runs\n")

for si, seed in enumerate(seeds):
    print(f"\n{'#'*65}")
    print(f"  SEED {seed}  ({si+1}/{len(seeds)})")
    print(f"{'#'*65}")

    # Recreate partitions with this seed
    part_iid = partition_iid(X_train, y_train, CFG["num_clients"], seed=seed)
    parts_dir = {}
    for alpha in CFG["dirichlet_alphas"]:
        parts_dir[alpha] = partition_dirichlet(
            y_train, CFG["num_clients"], alpha, seed=seed)

    for model_name in models:
        global CURRENT_MODEL
        CURRENT_MODEL = model_name

        print(f"\n  {'='*55}")
        print(f"  MODEL: {model_name}  |  SEED: {seed}")
        print(f"  {'='*55}")

        # Centralized baseline
        cent = run_centralized(seed=seed)
        cent["model"] = model_name
        cent["run_name"] = f"{model_name}_Centralized"
        all_results.append(cent)

        # IID baseline
        iid_r = run_fl_experiment(
            f"{model_name}_FedAvg_IID", "FedAvg", part_iid, "IID", seed=seed
        )
        iid_r["model"] = model_name
        all_results.append(iid_r)

        # 3×3 matrix
        algos = [
            ("FedAvg",        0.0,                0.0),
            ("FedProx",       CFG["fedprox_mu"],   0.0),
            ("FedTrimmedAvg", 0.0,                CFG["trimmed_fraction"]),
        ]
        for alpha in CFG["dirichlet_alphas"]:
            part = parts_dir[alpha]
            for algo_name, mu, trim in algos:
                rname = f"{model_name}_{algo_name}_a{alpha}"
                r = run_fl_experiment(
                    rname, algo_name, part, str(alpha),
                    proximal_mu=mu, trim_fraction=trim, seed=seed
                )
                r["model"] = model_name
                all_results.append(r)

print(f"\n{'='*65}")
print(f"  ALL {len(all_results)} RUNS COMPLETE")
print(f"  ({len(models)} models × {len(seeds)} seeds × 11 experiments)")
print(f"{'='*65}")

# Save raw results
raw_rows = []
for r in all_results:
    raw_rows.append(dict(
        Model=r.get("model", "MLP"),
        Run=r["run_name"], Algorithm=r["algorithm"], Alpha=r["alpha"],
        Seed=r["seed"],
        Final_Acc=r["final_acc"], Best_Acc=r["best_acc"],
        Final_F1=r["final_f1"], Best_F1=r["best_f1"],
        Conv_Round=r["convergence_round"],
    ))
df_raw = pd.DataFrame(raw_rows)
df_raw.to_csv(OUTPUT_DIR / "csv" / "all_runs_raw.csv", index=False)
print(f"\nRaw results saved: {OUTPUT_DIR / 'csv' / 'all_runs_raw.csv'}")


Running 2 model(s) × 3 seed(s) × 11 experiments = 66 runs


#################################################################
  SEED 42  (1/3)
#################################################################

  MODEL: MLP  |  SEED: 42

  CENTRALIZED BASELINE  |  seed=42
  Epoch   1/90  Loss:0.2145  Acc:0.8662  F1:0.8437
  Epoch  18/90  Loss:0.1611  Acc:0.8917  F1:0.8728
  Epoch  36/90  Loss:0.3109  Acc:0.8722  F1:0.8551
  Epoch  54/90  Loss:0.1469  Acc:0.8969  F1:0.8774
  Epoch  72/90  Loss:0.1449  Acc:0.8973  F1:0.8778
  Epoch  90/90  Loss:0.1430  Acc:0.8979  F1:0.8783

  MLP_FedAvg_IID  |  FedAvg  |  α=IID  |  seed=42
  Round  0 (init) |  Acc: 0.2862  F1: 0.2141  Loss: 2.1954
  Round  1/30  |  Acc: 0.8141  F1: 0.7888  Loss: 0.3233
  Round  5/30  |  Acc: 0.8738  F1: 0.8552  Loss: 0.2322
  Round 10/30  |  Acc: 0.8604  F1: 0.8434  Loss: 0.2665
  Round 15/30  |  Acc: 0.8930  F1: 0.8737  Loss: 0.1563
  Round 20/30  |  Acc: 0.8938  F1: 0.8745  Loss: 0.1534
  Round 25/30  |  Acc: 0.8948  F

## 10. Results and Analysis

### 10.1 Summary Table

In [ ]:
# ============================================================
# 10.1 — Results Summary (Mean ± Std across seeds, per model)
# ============================================================

grouped = df_raw.groupby(["Model", "Run", "Algorithm", "Alpha"]).agg(
    Acc_mean=("Best_Acc", "mean"), Acc_std=("Best_Acc", "std"),
    F1_mean=("Best_F1", "mean"), F1_std=("Best_F1", "std"),
    Conv_mean=("Conv_Round", "mean"), Conv_std=("Conv_Round", "std"),
    n_seeds=("Seed", "count"),
).reset_index().fillna(0)

display_rows = []
for _, row in grouped.iterrows():
    display_rows.append(dict(
        Model=row["Model"],
        Run=row["Run"],
        Algorithm=row["Algorithm"],
        Alpha=row["Alpha"],
        Best_Acc=f'{row["Acc_mean"]:.4f} \u00b1 {row["Acc_std"]:.4f}',
        Best_F1=f'{row["F1_mean"]:.4f} \u00b1 {row["F1_std"]:.4f}',
        Conv_Round=f'{row["Conv_mean"]:.1f} \u00b1 {row["Conv_std"]:.1f}',
        Seeds=int(row["n_seeds"]),
    ))

df_summary = pd.DataFrame(display_rows)
print(df_summary.to_string(index=False))

df_summary.to_csv(OUTPUT_DIR / "csv" / "experiment_summary_mean_std.csv", index=False)
grouped.to_csv(OUTPUT_DIR / "csv" / "experiment_summary_numeric.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR / 'csv'}")


Model                    Run     Algorithm Alpha        Best_Acc         Best_F1 Conv_Round  Seeds
  CNN        CNN_Centralized   Centralized     - 0.8981 ± 0.0000 0.8785 ± 0.0000  3.0 ± 1.7      3
  CNN         CNN_FedAvg_IID        FedAvg   IID 0.8958 ± 0.0010 0.8761 ± 0.0011  7.7 ± 0.6      3
  CNN        CNN_FedAvg_a0.1        FedAvg   0.1 0.8049 ± 0.0105 0.7821 ± 0.0146 24.3 ± 0.6      3
  CNN        CNN_FedAvg_a0.5        FedAvg   0.5 0.8938 ± 0.0025 0.8741 ± 0.0025  8.7 ± 1.5      3
  CNN        CNN_FedAvg_a1.0        FedAvg   1.0 0.8964 ± 0.0006 0.8767 ± 0.0007  7.3 ± 0.6      3
  CNN       CNN_FedProx_a0.1       FedProx   0.1 0.7412 ± 0.0017 0.6869 ± 0.0038 13.3 ± 1.2      3
  CNN       CNN_FedProx_a0.5       FedProx   0.5 0.8857 ± 0.0064 0.8652 ± 0.0070 13.7 ± 3.5      3
  CNN       CNN_FedProx_a1.0       FedProx   1.0 0.8933 ± 0.0014 0.8735 ± 0.0015 10.0 ± 3.0      3
  CNN CNN_FedTrimmedAvg_a0.1 FedTrimmedAvg   0.1 0.7989 ± 0.0284 0.7766 ± 0.0320 21.0 ± 2.6      3
  CNN CNN_

### 10.2 Convergence Plots

In [ ]:
# ============================================================
# 10.2 — Convergence Curves: MLP vs CNN (Mean ± Std Bands)
# ============================================================

from collections import defaultdict

models_in_results = sorted(set(r.get("model", "MLP") for r in all_results))
n_models = len(models_in_results)

fig, axes = plt.subplots(1, n_models, figsize=(8 * n_models, 6))
if n_models == 1:
    axes = [axes]

for ax, model_name in zip(axes, models_in_results):
    model_runs = [r for r in all_results if r.get("model") == model_name]

    # Group histories by experiment name (across seeds)
    history_groups = defaultdict(list)
    for r in model_runs:
        short = r["run_name"].replace(f"{model_name}_", "")
        history_groups[short].append(r["history"])

    colors = plt.cm.tab10.colors
    for idx, (name, histories) in enumerate(sorted(history_groups.items())):
        min_len = min(len(h["f1"]) for h in histories)
        f1_matrix = np.array([h["f1"][:min_len] for h in histories])
        rounds = np.arange(1, min_len + 1)
        f1_mean = f1_matrix.mean(axis=0)
        f1_std = f1_matrix.std(axis=0)

        ls = "--" if "Centralized" in name else ("-." if "IID" in name else "-")
        c = colors[idx % len(colors)]
        ax.plot(rounds, f1_mean, label=name, linestyle=ls, color=c, marker="o", ms=2)
        ax.fill_between(rounds, f1_mean - f1_std, f1_mean + f1_std, alpha=0.15, color=c)

    ax.set_title(f"{model_name} — Macro F1 (Mean ± Std)", fontweight="bold")
    ax.set_xlabel("Round / Epoch")
    ax.set_ylabel("Macro F1")
    ax.legend(fontsize=6, loc="lower right")
    ax.grid(alpha=0.3)
    ax.set_ylim(0.40, 0.92)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "convergence_mlp_vs_cnn.png", dpi=200, bbox_inches="tight")
plt.show()
print("✅ MLP vs CNN convergence curves saved.")


✅ MLP vs CNN convergence curves saved.


### 10.3 Algorithm Comparison by Non-IID Level

In [ ]:
# ============================================================
# 10.3 — Algorithm × Model Comparison with Error Bars
# ============================================================

matrix_g = grouped[~grouped["Alpha"].isin(["-", "IID"])].copy()

if len(matrix_g) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, (mean_col, std_col, title) in zip(axes, [
        ("Acc_mean", "Acc_std", "Best Accuracy"),
        ("F1_mean", "F1_std", "Best Macro F1"),
        ("Conv_mean", "Conv_std", "Convergence Round"),
    ]):
        # Build bars: one group per (model, algorithm)
        models_list = sorted(matrix_g["Model"].unique())
        algos_order = ["FedAvg", "FedProx", "FedTrimmedAvg"]
        alphas_order = sorted(matrix_g["Alpha"].unique())
        x = np.arange(len(alphas_order))

        bar_groups = []
        for model in models_list:
            for algo in algos_order:
                bar_groups.append((model, algo))

        n_bars = len(bar_groups)
        width = 0.8 / n_bars

        for i, (model, algo) in enumerate(bar_groups):
            subset = matrix_g[(matrix_g["Model"] == model) & (matrix_g["Algorithm"] == algo)]
            means = [subset[subset["Alpha"] == a][mean_col].values[0]
                     if len(subset[subset["Alpha"] == a]) > 0 else 0
                     for a in alphas_order]
            stds = [subset[subset["Alpha"] == a][std_col].values[0]
                    if len(subset[subset["Alpha"] == a]) > 0 else 0
                    for a in alphas_order]
            offset = (i - n_bars / 2 + 0.5) * width
            label = f"{model}_{algo}"
            ax.bar(x + offset, means, width, yerr=stds,
                   label=label, capsize=3, alpha=0.85)

        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Non-IID Level (α)")
        ax.set_ylabel(title)
        ax.set_xticks(x)
        ax.set_xticklabels([f"α={a}" for a in alphas_order])
        ax.legend(fontsize=6, ncol=2)
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plots" / "comparison_mlp_vs_cnn_bars.png",
                dpi=200, bbox_inches="tight")
    plt.show()
    print("✅ MLP vs CNN comparison bars saved.")


✅ MLP vs CNN comparison bars saved.


## 11. Publication Exports

### 11.1 LaTeX Table

In [ ]:
# ============================================================
# 11.1 — LaTeX Table (Mean ± Std, with Model column)
# ============================================================

ltx_rows = []
for _, row in grouped.iterrows():
    ltx_rows.append({
        "Model": row["Model"],
        "Algorithm": row["Algorithm"],
        r"$\alpha$": row["Alpha"],
        r"Acc (\%)": f'${row["Acc_mean"]*100:.2f} \\pm {row["Acc_std"]*100:.2f}$',
        r"F1 (\%)": f'${row["F1_mean"]*100:.2f} \\pm {row["F1_std"]*100:.2f}$',
        "Conv.": f'${row["Conv_mean"]:.1f} \\pm {row["Conv_std"]:.1f}$',
    })

df_ltx = pd.DataFrame(ltx_rows)
latex_str = df_ltx.to_latex(
    index=False, escape=False, column_format="llcrrr",
    caption=(
        "Comparison of federated aggregation strategies on N-BaIoT "
        "using MLP and 1D-CNN architectures under varying non-IID conditions. "
        f"Results reported as mean $\\pm$ std over {len(CFG['seeds'])} seeds."
    ),
    label="tab:fl_results",
)

with open(OUTPUT_DIR / "latex" / "results_table.tex", "w") as f:
    f.write(latex_str)
print(latex_str)
print(f"\nSaved to {OUTPUT_DIR / 'latex' / 'results_table.tex'}")


\begin{table}
\caption{Comparison of federated aggregation strategies on N-BaIoT using MLP and 1D-CNN architectures under varying non-IID conditions. Results reported as mean $\pm$ std over 3 seeds.}
\label{tab:fl_results}
\begin{tabular}{llcrrr}
\toprule
Model & Algorithm & $\alpha$ & Acc (\%) & F1 (\%) & Conv. \\
\midrule
CNN & Centralized & - & $89.81 \pm 0.00$ & $87.85 \pm 0.00$ & $3.0 \pm 1.7$ \\
CNN & FedAvg & IID & $89.58 \pm 0.10$ & $87.61 \pm 0.11$ & $7.7 \pm 0.6$ \\
CNN & FedAvg & 0.1 & $80.49 \pm 1.05$ & $78.21 \pm 1.46$ & $24.3 \pm 0.6$ \\
CNN & FedAvg & 0.5 & $89.38 \pm 0.25$ & $87.41 \pm 0.25$ & $8.7 \pm 1.5$ \\
CNN & FedAvg & 1.0 & $89.64 \pm 0.06$ & $87.67 \pm 0.07$ & $7.3 \pm 0.6$ \\
CNN & FedProx & 0.1 & $74.12 \pm 0.17$ & $68.69 \pm 0.38$ & $13.3 \pm 1.2$ \\
CNN & FedProx & 0.5 & $88.57 \pm 0.64$ & $86.52 \pm 0.70$ & $13.7 \pm 3.5$ \\
CNN & FedProx & 1.0 & $89.33 \pm 0.14$ & $87.35 \pm 0.15$ & $10.0 \pm 3.0$ \\
CNN & FedTrimmedAvg & 0.1 & $79.89 \pm 2.84$ & $77.66 \p

### 11.2 Auto-Generated Findings Summary

In [ ]:
# ============================================================
# 11.2 — Auto-Generated Findings (Mean ± Std, per model)
# ============================================================

n_seeds = len(CFG["seeds"])
task_desc = ("IoT device identification (device-as-class fallback)"
             if USE_DEVICE_AS_CLASS
             else "IoT botnet traffic detection (benign vs attack types)")

findings = f"## Auto-Generated Findings (Mean ± Std over {n_seeds} seeds)\n\n"
findings += f"**Task:** {task_desc}\n"
findings += f"**Models:** {', '.join(CFG['models'])}\n"
findings += f"**All values computed at runtime.**\n\n"

for model_name in CFG["models"]:
    mg = grouped[grouped["Model"] == model_name]
    mx = mg[~mg["Alpha"].isin(["-", "IID"])]
    if len(mx) == 0:
        continue

    best_r = mx.loc[mx["F1_mean"].idxmax()]
    worst_r = mx.loc[mx["F1_mean"].idxmin()]
    fast_r = mx.loc[mx["Conv_mean"].idxmin()]
    c = mg[mg["Algorithm"] == "Centralized"]
    c_f1 = c.iloc[0]["F1_mean"] if len(c) > 0 else 0
    c_f1_std = c.iloc[0]["F1_std"] if len(c) > 0 else 0
    iid = mg[mg["Alpha"] == "IID"]
    iid_f1 = iid.iloc[0]["F1_mean"] if len(iid) > 0 else 0

    findings += f"### {model_name} Architecture\n\n"
    findings += f"**Centralized:** F1 = {c_f1*100:.2f}% ± {c_f1_std*100:.2f}%\n\n"
    findings += f"**IID Baseline:** F1 = {iid_f1*100:.2f}%  |  Gap vs centralized: {(c_f1-iid_f1)*100:+.2f}pp\n\n"
    findings += (f"**Best Non-IID:** {best_r['Run']} — "
                 f"F1 = {best_r['F1_mean']*100:.2f}% ± {best_r['F1_std']*100:.2f}%\n\n")
    findings += (f"**Worst Non-IID:** {worst_r['Run']} — "
                 f"F1 = {worst_r['F1_mean']*100:.2f}% ± {worst_r['F1_std']*100:.2f}%\n\n")

    # Per-algorithm degradation
    findings += "**Non-IID degradation (α=1.0 → α=0.1):**\n\n"
    for algo in ["FedAvg", "FedProx", "FedTrimmedAvg"]:
        a10 = mx[(mx["Algorithm"] == algo) & (mx["Alpha"] == "1.0")]
        a01 = mx[(mx["Algorithm"] == algo) & (mx["Alpha"] == "0.1")]
        if len(a10) > 0 and len(a01) > 0:
            drop = (a10.iloc[0]["F1_mean"] - a01.iloc[0]["F1_mean"]) * 100
            findings += f"- {algo}: {drop:.2f}pp drop\n"
    findings += "\n---\n\n"

# Cross-architecture comparison
findings += "### Cross-Architecture Comparison\n\n"
for alpha in ["1.0", "0.5", "0.1"]:
    findings += f"**α = {alpha}:**\n"
    sub = grouped[(grouped["Alpha"] == alpha) & (~grouped["Alpha"].isin(["-", "IID"]))]
    sub = sub.sort_values("F1_mean", ascending=False)
    for _, row in sub.iterrows():
        findings += f"- {row['Model']}_{row['Algorithm']}: F1 = {row['F1_mean']*100:.2f}% ± {row['F1_std']*100:.2f}%\n"
    findings += "\n"

print(findings)
with open(OUTPUT_DIR / "csv" / "findings_summary.md", "w", encoding="utf-8") as f:
    f.write(findings)
print(f"Saved to {OUTPUT_DIR / 'csv' / 'findings_summary.md'}")


## Auto-Generated Findings (Mean ± Std over 3 seeds)

**Task:** IoT botnet traffic detection (benign vs attack types)
**Models:** MLP, CNN
**All values computed at runtime.**

### MLP Architecture

**Centralized:** F1 = 87.84% ± 0.00%

**IID Baseline:** F1 = 87.75%  |  Gap vs centralized: +0.09pp

**Best Non-IID:** MLP_FedTrimmedAvg_a1.0 — F1 = 87.71% ± 0.08%

**Worst Non-IID:** MLP_FedProx_a0.1 — F1 = 76.48% ± 6.41%

**Non-IID degradation (α=1.0 → α=0.1):**

- FedAvg: 6.41pp drop
- FedProx: 10.84pp drop
- FedTrimmedAvg: 4.37pp drop

---

### CNN Architecture

**Centralized:** F1 = 87.85% ± 0.00%

**IID Baseline:** F1 = 87.61%  |  Gap vs centralized: +0.24pp

**Best Non-IID:** CNN_FedAvg_a1.0 — F1 = 87.67% ± 0.07%

**Worst Non-IID:** CNN_FedProx_a0.1 — F1 = 68.69% ± 0.38%

**Non-IID degradation (α=1.0 → α=0.1):**

- FedAvg: 9.46pp drop
- FedProx: 18.65pp drop
- FedTrimmedAvg: 9.95pp drop

---

### Cross-Architecture Comparison

**α = 1.0:**
- MLP_FedTrimmedAvg: F1 = 87.71% ± 0.08%
- ML

## 12. Export All Outputs

In [ ]:
# ============================================================
# 12.1 — Download Archive
# ============================================================

shutil.make_archive("fl_nbaiot_results", "zip", OUTPUT_DIR)
print("📦 Archive: fl_nbaiot_results.zip")
print("\nContents:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), "").count(os.sep)
    print(f"  {"  " * level}{os.path.basename(root)}/")
    for f in sorted(files):
        sz = os.path.getsize(os.path.join(root, f))
        print(f"  {"  " * (level+1)}{f}  ({sz/1024:.1f} KB)")

try:
    from google.colab import files
    files.download("fl_nbaiot_results.zip")
    print("\n⬇️ Download started.")
except ImportError:
    print(f"\nArchive at: {os.path.abspath('fl_nbaiot_results.zip')}")

---
## Research Integrity Notice
> **All numerical results, plots, and tables in this notebook are generated at runtime.** No metrics are manually entered. The notebook is fully reproducible with the specified seeds (42, 123, 7). Both MLP and 1D-CNN architectures are evaluated under identical conditions. Every figure includes error bars or std bands from multi-seed validation.
